# AudioMAE - Smoke Test

Generates a synthetic 10 second WAV, converts it to a Kaldi-compatible Mel fbank, loads the official fine-tuned checkpoint, and runs one forward pass.

In [ ]:
import json
import math
import sys
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
import torchaudio

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "repo").is_dir():
    raise RuntimeError("Launch this notebook from experiments/audiomae/notebooks or experiments/audiomae")
REPO = ROOT / "repo"
sys.path.insert(0, str(REPO))

import util.misc as misc
import models_vit
from main_finetune_as import PatchEmbed_new

In [ ]:
smoke_dir = ROOT / "data" / "smoke"
smoke_dir.mkdir(parents=True, exist_ok=True)
wav_path = smoke_dir / "synthetic_10s_16k.wav"

sr = 16000
t = np.arange(sr * 10, dtype=np.float32) / sr
wave = 0.2 * np.sin(2 * np.pi * 440 * t) + 0.1 * np.sin(2 * np.pi * 880 * t)
sf.write(str(wav_path), wave, sr)

loaded, loaded_sr = sf.read(str(wav_path), dtype="float32")
waveform = torch.from_numpy(loaded).unsqueeze(0)
fbank = torchaudio.compliance.kaldi.fbank(
    waveform - waveform.mean(),
    htk_compat=True,
    sample_frequency=loaded_sr,
    use_energy=False,
    window_type="hanning",
    num_mel_bins=128,
    dither=0.0,
    frame_shift=10,
)

target_length = 1024
if fbank.shape[0] < target_length:
    fbank = torch.nn.ZeroPad2d((0, 0, 0, target_length - fbank.shape[0]))(fbank)
else:
    fbank = fbank[:target_length]

fbank = (fbank - (-4.2677393)) / (4.5689974 * 2)
sample = fbank.unsqueeze(0).unsqueeze(0)
print("waveform:", tuple(waveform.shape))
print("fbank:", tuple(fbank.shape))
print("sample:", tuple(sample.shape))

In [ ]:
model = models_vit.vit_base_patch16(
    num_classes=527,
    drop_path_rate=0.1,
    global_pool=True,
    mask_2d=True,
    use_custom_patch=False,
)
model.patch_embed = PatchEmbed_new(
    img_size=(1024, 128),
    patch_size=(16, 16),
    in_chans=1,
    embed_dim=768,
    stride=16,
)
model.pos_embed = torch.nn.Parameter(
    torch.zeros(1, model.patch_embed.num_patches + 1, 768),
    requires_grad=False,
)

ckpt_path = ROOT / "ckpt" / "finetuned.pth"
ckpt = misc.torch_load_compat(str(ckpt_path), map_location="cpu")
msg = model.load_state_dict(ckpt["model"], strict=False)
print("checkpoint_epoch:", ckpt.get("epoch"))
print("missing_keys:", msg.missing_keys)
print("unexpected_keys:", msg.unexpected_keys)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()
with torch.no_grad():
    out = model(sample.to(device))

report = {
    "wav": str(wav_path),
    "fbank_shape": list(fbank.shape),
    "sample_shape": list(sample.shape),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "checkpoint_epoch": int(ckpt.get("epoch", -1)),
    "missing_keys": list(msg.missing_keys),
    "unexpected_keys": list(msg.unexpected_keys),
    "output_shape": list(out.shape),
    "output_mean": float(out.mean().detach().cpu()),
    "output_std": float(out.std().detach().cpu()),
}

(ROOT / "logs").mkdir(exist_ok=True)
(ROOT / "logs" / "smoke_test.txt").write_text(json.dumps(report, indent=2), encoding="utf-8")
report